# Generalised APO — Multi-Model Clinical Summarization
Automatic Prompt Optimization (critique → patch → 3 candidates → best-by-reward), generalised across the same 4 models as the ISP & zero-shot notebooks.

**Switch models by changing only `SELECTED_MODEL` in Cell 3.** Supported: `qwen2.5`, `ministral_8b`, `llama3.1_8b`, `qwen3`.

Matched to ISP/zero-shot: fixed 150 tokens (single-pass), BERTScore = roberta-large (no baseline), reward = 0.6·ROUGE-L + 0.4·BERT-F1 (= summarisation score), summary grammar per case + prompt grammar per prompt. Phase-2 CSV columns match the ISP final CSV.

Phase 1: APO loop on N=150 pairs, K=3 iterations, 3 candidates/iter. Phase 2: full evaluation on all 592 pairs with the best prompt.

In [ ]:
# ── Cell 1 · Install ─────────────────────────────────────────
import subprocess
subprocess.run(["pip","install","-q","transformers","accelerate","bitsandbytes",
    "datasets","evaluate","rouge-score","bert-score","pandas","matplotlib","seaborn",
    "language_tool_python==2.9.2","nltk","sentencepiece"], check=True)
subprocess.run(["pip","install","-q","-U","transformers"], check=True)
print("Dependencies installed")

In [ ]:
# ── Cell 2 · Imports ─────────────────────────────────────────
import os, json, re, gc, random, textwrap
from pathlib import Path
from dataclasses import dataclass
from typing import List, Dict, Tuple
import torch, numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from transformers import (AutoTokenizer, AutoModelForCausalLM,
    AutoModelForSequenceClassification, BitsAndBytesConfig)
from rouge_score import rouge_scorer as rouge_scorer_lib
from bert_score import score as bert_score_fn
import nltk, language_tool_python
nltk.download("punkt", quiet=True); nltk.download("punkt_tab", quiet=True)
print("Imports ok | CUDA:", torch.cuda.is_available())

In [ ]:
# ── Cell 3 · Config, model registry & selection ──────────────
# ╔══════════════════════════════════════════════════════════════╗
# ║  CHANGE ONLY THIS LINE TO SWITCH MODELS                       ║
# ╚══════════════════════════════════════════════════════════════╝
SELECTED_MODEL = "qwen2.5"   # qwen2.5 | ministral_8b | llama3.1_8b | qwen3

MODEL_REGISTRY = {
    "qwen2.5":      {"repo": "Qwen/Qwen2.5-7B-Instruct",            "gated": False, "dtype": torch.float16,  "display": "Qwen2.5-7B"},
    "ministral_8b": {"repo": "mistralai/Ministral-8B-Instruct-2410", "gated": True,  "dtype": torch.bfloat16, "display": "Ministral-8B"},
    "llama3.1_8b":  {"repo": "meta-llama/Llama-3.1-8B-Instruct",     "gated": True,  "dtype": torch.bfloat16, "display": "Llama-3.1-8B"},
    "qwen3":        {"repo": "Qwen/Qwen3-8B",                        "gated": False, "dtype": torch.bfloat16, "display": "Qwen3-8B"},
}
assert SELECTED_MODEL in MODEL_REGISTRY
_cfg = MODEL_REGISTRY[SELECTED_MODEL]

# fp16 fastest on all GPUs (bf16 slow on T4)
FORCE_FP16 = True
if FORCE_FP16: _DTYPE = torch.float16
elif torch.cuda.is_available() and torch.cuda.is_bf16_supported(): _DTYPE = _cfg["dtype"]
else: _DTYPE = torch.float16

@dataclass
class Config:
    data_root:     str = "/kaggle/input/biolaysumm/Data/Short"
    fulltext_dir:  str = "fulltext"
    summaries_dir: str = "summaries"
    output_dir:    str = "/kaggle/working"
    graphs_dir:    str = "/kaggle/working/graphs"

    generator_model:  str = _cfg["repo"]
    model_tag:        str = SELECTED_MODEL
    model_display:    str = _cfg["display"]
    model_gated:      bool = _cfg["gated"]
    compute_dtype:    object = _DTYPE
    bert_score_model: str = "roberta-large"   # match ISP (no baseline rescale)
    nli_model:        str = "cross-encoder/nli-deberta-v3-large"

    n_apo_pairs:  int = 150
    n_iterations: int = 3
    n_candidates: int = 3
    alpha: float = 0.60   # ROUGE-L weight  (reward = summarisation score)
    beta:  float = 0.40   # BERTScore-F1 weight

    fixed_tokens: int = 150   # match ISP/zero-shot; single-pass, no retry
    seed: int = 42

    # sampling
    temperature: float = 0.1
    top_p:       float = 0.9

cfg = Config()
random.seed(cfg.seed); np.random.seed(cfg.seed); torch.manual_seed(cfg.seed)
Path(cfg.output_dir).mkdir(parents=True, exist_ok=True)
Path(cfg.graphs_dir).mkdir(parents=True, exist_ok=True)

def OUT(name): return str(Path(cfg.output_dir) / f"{cfg.model_tag}_{name}")

W_ROUGEL, W_BERT = cfg.alpha, cfg.beta   # summarisation-score weights

NLI_CONTRADICTION, NLI_ENTAILMENT, NLI_NEUTRAL = 0, 1, 2
NLI_LABEL_MAP = {NLI_CONTRADICTION:"contradiction", NLI_ENTAILMENT:"entailment", NLI_NEUTRAL:"neutral"}

print("Model:", cfg.generator_model, "| tag:", cfg.model_tag, "| gated:", cfg.model_gated)
print("dtype:", cfg.compute_dtype, "| fixed_tokens:", cfg.fixed_tokens)
print("reward = %.1f*ROUGE-L + %.1f*BERT-F1  (= summarisation score)" % (cfg.alpha, cfg.beta))

In [ ]:
# ── Cell 4 · HuggingFace token (gated models) ────────────────
HF_TOKEN_FALLBACK = ""   # optionally paste hf_xxx
HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN"); print("HF token from Kaggle Secrets")
except Exception: pass
if not HF_TOKEN:
    HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
    if HF_TOKEN: print("HF token from environment")
if not HF_TOKEN and HF_TOKEN_FALLBACK:
    HF_TOKEN = HF_TOKEN_FALLBACK; print("HF token from fallback")
if HF_TOKEN:
    os.environ["HF_TOKEN"]=HF_TOKEN; os.environ["HUGGING_FACE_HUB_TOKEN"]=HF_TOKEN
    try:
        from huggingface_hub import login; login(token=HF_TOKEN, add_to_git_credential=False)
    except Exception as e: print("hub login skipped:", e)
else:
    print("No HF token (open models work; gated Llama/Ministral will fail).")
if cfg.model_gated and not HF_TOKEN:
    raise RuntimeError(f"'{SELECTED_MODEL}' is gated — set HF_TOKEN and request access to {cfg.generator_model}.")
HF_AUTH = {"token": HF_TOKEN} if HF_TOKEN else {}

In [ ]:
# ── Cell 5 · Data loading ────────────────────────────────────
def load_pairs(cfg: Config) -> List[Dict]:
    ft_path  = Path(cfg.data_root) / cfg.fulltext_dir
    sum_path = Path(cfg.data_root) / cfg.summaries_dir
    pairs = []
    for ft_file in sorted(ft_path.glob("*.txt")):
        stem = ft_file.stem; sum_file = sum_path / f"{stem}_sum.txt"
        if not sum_file.exists(): continue
        ft = ft_file.read_text(encoding="utf-8", errors="ignore").strip()
        sm = sum_file.read_text(encoding="utf-8", errors="ignore").strip()
        if ft and sm: pairs.append({"filename": stem, "fulltext": ft, "summary": sm})
    print(f"Loaded {len(pairs)} pairs")
    return pairs

def select_apo_pairs(all_pairs: List[Dict], n: int, seed: int) -> List[Dict]:
    rng = random.Random(seed); idx = list(range(len(all_pairs)))
    rng.shuffle(idx); chosen = sorted(idx[:min(n, len(all_pairs))])
    sub = [all_pairs[i] for i in chosen]
    print(f"APO subset: {len(sub)} pairs")
    return sub

In [ ]:
# ── Cell 6 · Model loading, chat template, grammar ───────────
def load_generator(cfg: Config):
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=cfg.compute_dtype)
    tok = AutoTokenizer.from_pretrained(cfg.generator_model, trust_remote_code=True, **HF_AUTH)
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    mdl = AutoModelForCausalLM.from_pretrained(cfg.generator_model, quantization_config=bnb,
        device_map="auto", trust_remote_code=True, low_cpu_mem_usage=True, **HF_AUTH)
    mdl.eval()
    print(f"Generator loaded: {cfg.generator_model}")
    return tok, mdl

def load_nli(cfg: Config):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    tok = AutoTokenizer.from_pretrained(cfg.nli_model)
    mdl = AutoModelForSequenceClassification.from_pretrained(cfg.nli_model).to(device).eval()
    print(f"NLI loaded on {device}: {cfg.nli_model} | id2label: {mdl.config.id2label}")
    return tok, mdl, device

def build_chat_text(tok, messages):
    """Chat template; disable Qwen3 thinking mode where supported."""
    try:
        return tok.apply_chat_template(messages, tokenize=False,
                                       add_generation_prompt=True, enable_thinking=False)
    except TypeError:
        return tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

# ── Grammar scoring (same loaded model + LanguageTool fallback) ──
_LT = language_tool_python.LanguageTool("en-US")
def _complete_grammar(tok, mdl, messages, max_tokens=5):
    text = build_chat_text(tok, messages)
    mi = tok([text], return_tensors="pt").to(mdl.device)
    with torch.no_grad():
        gi = mdl.generate(**mi, max_new_tokens=max_tokens, temperature=0.1, do_sample=False,
                          pad_token_id=tok.pad_token_id, eos_token_id=tok.eos_token_id)
    return tok.decode(gi[0][len(mi.input_ids[0]):], skip_special_tokens=True).strip("\n").strip()
def _lt_fallback(text):
    score=100
    for m in _LT.check(text):
        if m.ruleId.startswith("MORFOLOGIK_") or m.ruleId=="UPPERCASE_SENTENCE_START": score-=5
        elif "AGREEMENT" in m.ruleId: score-=15
        elif "GRAMMAR" in m.ruleId or "SENTENCE" in m.ruleId: score-=20
        else: score-=10
    return max(0, score)
def get_llm_grammar_score(text, tok, mdl):
    if not text or not text.strip(): return 0
    messages=[{"role":"system","content":"You are a strict grammar evaluator. Score grammar from 0 to 100. 100 = perfect grammar. 0 = very poor grammar. Return only a number, no text."},
              {"role":"user","content":'Evaluate the grammar of this text and return a score from 0 to 100.\nText:\n"""\n'+text+'\n"""'}]
    try:
        raw=_complete_grammar(tok, mdl, messages, 5)
        mt=re.match(r"^\[?(\d+)\]?$", raw.strip())
        if mt: score=int(mt.group(1))
        else:
            nums=re.findall(r"\d+", raw); score=int(nums[0]) if nums else None
            if score is None: raise ValueError("no number")
        return score if 0<=score<=100 else _lt_fallback(text)
    except (ValueError, TypeError):
        return _lt_fallback(text)
    except RuntimeError as e:
        if "CUDA out of memory" in str(e):
            torch.cuda.empty_cache(); gc.collect(); return _lt_fallback(text)
        raise

In [ ]:
# ── Cell 7 · Generation (fixed 150, single-pass) + metrics ───
def generate_summary(fulltext, reference, prompt_template, gen_tok, gen_mdl, cfg):
    """Single-pass, fixed 150 tokens. Returns (text, gen_tok_count, is_complete)."""
    messages = [{"role": "user", "content": prompt_template.format(fulltext=fulltext)}]
    text = build_chat_text(gen_tok, messages)
    inputs = gen_tok([text], return_tensors="pt", truncation=True, max_length=4096).to(gen_mdl.device)
    with torch.no_grad():
        out = gen_mdl.generate(**inputs,
            max_new_tokens=cfg.fixed_tokens, min_new_tokens=cfg.fixed_tokens,
            temperature=cfg.temperature, top_p=cfg.top_p, do_sample=True,
            repetition_penalty=1.2,
            pad_token_id=gen_tok.pad_token_id, eos_token_id=gen_tok.eos_token_id)
    gen_ids = out[0][inputs["input_ids"].shape[1]:]
    decoded = gen_tok.decode(gen_ids, skip_special_tokens=True).strip()
    # trim to last full stop
    t = decoded.strip(); stops = list(re.finditer(r"[.!?]", t))
    trimmed = t[:stops[-1].end()].strip() if stops else t
    if not trimmed: trimmed = decoded if decoded else "Summary unavailable."
    gen_tc = len(gen_tok(trimmed, add_special_tokens=False)["input_ids"])
    is_complete = bool(re.search(r"[.!?]\s*$", trimmed))
    return trimmed, gen_tc, is_complete

_rouge = rouge_scorer_lib.RougeScorer(["rouge1","rouge2","rougeL"], use_stemmer=True)
def compute_rouge(hyp, ref):
    s = _rouge.score(ref, hyp)
    return {"rouge1": s["rouge1"].fmeasure, "rouge2": s["rouge2"].fmeasure, "rougeL": s["rougeL"].fmeasure}

def compute_bertscore_batch(hyps, refs, cfg, device="cpu"):
    P,R,F = bert_score_fn(hyps, refs, model_type=cfg.bert_score_model, lang="en",
                          device=device, verbose=False)   # no baseline rescale (match ISP)
    return {"bert_precision":P.tolist(), "bert_recall":R.tolist(), "bert_f1":F.tolist()}

def compute_entailment_batch(premises, hyps, nli_tok, nli_mdl, device, batch_size=16):
    results=[]
    for i in range(0, len(premises), batch_size):
        enc = nli_tok(premises[i:i+batch_size], hyps[i:i+batch_size], return_tensors="pt",
                      truncation=True, max_length=512, padding=True).to(device)
        with torch.no_grad(): logits = nli_mdl(**enc).logits
        for row in torch.softmax(logits, dim=-1).cpu():
            results.append({"entailed_prob":row[NLI_ENTAILMENT].item(),
                            "neutral_prob":row[NLI_NEUTRAL].item(),
                            "contradiction_prob":row[NLI_CONTRADICTION].item(),
                            "label":NLI_LABEL_MAP[int(row.argmax())]})
    return results

In [ ]:
# ── Cell 8 · Score a prompt on the APO subset ────────────────
def evaluate_prompt(prompt_template, pairs, gen_tok, gen_mdl, cfg):
    generated, references = [], []
    for pair in pairs:
        g,_,_ = generate_summary(pair["fulltext"], pair["summary"], prompt_template, gen_tok, gen_mdl, cfg)
        generated.append(g); references.append(pair["summary"])
    rouge_rows = [compute_rouge(g, r) for g,r in zip(generated, references)]
    bs = compute_bertscore_batch(generated, references, cfg, device="cpu")
    r1=[s["rouge1"] for s in rouge_rows]; r2=[s["rouge2"] for s in rouge_rows]
    rl=[s["rougeL"] for s in rouge_rows]; bf=bs["bert_f1"]
    rewards=[cfg.alpha*r + cfg.beta*b for r,b in zip(rl,bf)]   # = summarisation score
    return {"mean_rouge1":float(np.mean(r1)), "mean_rouge2":float(np.mean(r2)),
            "mean_rouge_l":float(np.mean(rl)), "mean_bert_f1":float(np.mean(bf)),
            "mean_reward":float(np.mean(rewards)),
            "_generated":generated, "_references":references,
            "_r1":r1,"_r2":r2,"_rl":rl,"_bf":bf,"_rewards":rewards}

In [ ]:
# ── Cell 9 · Critique & Patch (system prompts verbatim) ──────
CRITIQUE_SYSTEM = textwrap.dedent("""
    You are an expert evaluator of prompts used for biomedical lay summarization.
    You will receive:
    1. A prompt template currently used to generate summaries of clinical cases.
    2. Evaluation scores (ROUGE-1, ROUGE-2, ROUGE-L, BERTScore-F1) averaged
       over 150 clinical case pairs.
    3. Score statistics: mean, min, max for each metric.

    Your task:
    Write a detailed critique that identifies:
    - What aspects of the prompt are working well (high scores)
    - What specific weaknesses are causing low ROUGE or BERTScore
    - Whether the prompt gives clear enough instructions on:
        * length and detail level
        * medical terminology simplification
        * structure and focus (diagnosis, findings, outcome)
    - Any ambiguities or missing instructions in the prompt

    Be specific and constructive. Your critique will be used to rewrite
    the prompt, so focus on actionable observations.
""").strip()

PATCH_SYSTEM = textwrap.dedent("""
    You are an expert prompt engineer for biomedical lay summarization.
    You will receive:
    1. The current prompt template (it contains the placeholder {fulltext})
    2. A critique identifying its weaknesses
    3. A STRATEGY describing the angle your redesign must take

    Your task:
    DESIGN ONE new prompt template that fixes the critique's weaknesses using
    the given STRATEGY. This must be a genuine redesign, NOT a paraphrase or a
    light rewording of the current prompt: you may change the wording, the
    structure, the ordering of instructions, and the level of specificity.
    Rethink the task from scratch for this strategy.

    Hard rules:
    - Output ONLY the prompt template text. No preamble, no explanation, no
      title, no quotes, no commentary, no markdown fences.
    - The template MUST contain the exact placeholder {fulltext} somewhere,
      written literally as {fulltext} (single curly braces).
    - It must be a complete, self-contained instruction that works for ANY
      clinical case (never hard-code details of one specific case).
    - It must be clearly different from the current prompt.
""").strip()

# Distinct strategies, one per candidate (cycled if n_candidates changes).
PATCH_STRATEGIES = [
    "DETAIL / COVERAGE: explicitly enumerate every clinical element that must "
    "appear (demographics, presenting complaint, history, key findings and "
    "investigations, diagnosis, treatment, outcome) and insist none are omitted.",
    "READABILITY / SIMPLIFICATION: prioritise plain, accessible language for a "
    "lay audience -- define or avoid jargon, prefer short clear sentences, keep "
    "it faithful but easy to read.",
    "STRUCTURE / FLOW: impose a clear logical ordering (presentation -> findings "
    "-> diagnosis -> treatment -> outcome) and a tight, well-organised narrative "
    "in continuous prose.",
]

_PLACEHOLDER_RE = re.compile(r"\{+\s*fulltext\s*\}+|\[+\s*fulltext\s*\]+|<+\s*fulltext\s*>+",
                             flags=re.IGNORECASE)

def _normalize_prompt(text):
    """Repair a candidate so it is usable: strip fences/quotes, fix the
    {fulltext} placeholder variants, and ensure the placeholder is present."""
    t = text.strip()
    t = re.sub(r"^```[a-zA-Z]*\n?", "", t).strip()
    t = re.sub(r"\n?```$", "", t).strip()
    # strip a single pair of wrapping quotes (either kind)
    if len(t) >= 2 and t[0] in chr(34)+chr(39) and t[-1] == t[0]:
        t = t[1:-1].strip()
    t = _PLACEHOLDER_RE.sub("{fulltext}", t)
    if "{fulltext}" not in t:
        t = t.rstrip() + "\n\nClinical case:\n{fulltext}\n\nSummary:"
    # protect {fulltext}, escape any other stray braces so .format() is safe
    SENT = "__FULLTEXT_PH__"
    t = t.replace("{fulltext}", SENT)
    t = t.replace("{", "{{").replace("}", "}}")
    t = t.replace(SENT, "{fulltext}")
    return t.strip()

def run_critique(current_prompt, scores, gen_tok, gen_mdl):
    rl,bf,r1,r2 = scores["_rl"],scores["_bf"],scores["_r1"],scores["_r2"]
    stats_block = textwrap.dedent(f"""
        Evaluation over 150 pairs:
          ROUGE-1  : mean={scores['mean_rouge1']:.4f}  min={min(r1):.4f}  max={max(r1):.4f}
          ROUGE-2  : mean={scores['mean_rouge2']:.4f}  min={min(r2):.4f}  max={max(r2):.4f}
          ROUGE-L  : mean={scores['mean_rouge_l']:.4f}  min={min(rl):.4f}  max={max(rl):.4f}
          BERTScore: mean={scores['mean_bert_f1']:.4f}  min={min(bf):.4f}  max={max(bf):.4f}
          Combined reward: {scores['mean_reward']:.4f}
    """).strip()
    user_msg = textwrap.dedent(f"""
        Current prompt template:
        ---
        {current_prompt}
        ---

        {stats_block}

        Please write a detailed critique of this prompt based on the scores above.
    """).strip()
    messages=[{"role":"system","content":CRITIQUE_SYSTEM},{"role":"user","content":user_msg}]
    text=build_chat_text(gen_tok, messages)
    inputs=gen_tok([text], return_tensors="pt").to(gen_mdl.device)
    with torch.no_grad():
        out=gen_mdl.generate(**inputs, max_new_tokens=512, temperature=0.7, top_p=0.9,
                             do_sample=True, pad_token_id=gen_tok.pad_token_id, eos_token_id=gen_tok.eos_token_id)
    return gen_tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def _generate_one_candidate(current_prompt, critique, strategy, gen_tok, gen_mdl):
    user_msg = textwrap.dedent(f"""
        Current prompt template:
        ---
        {current_prompt}
        ---

        Critique of its weaknesses:
        ---
        {critique}
        ---

        STRATEGY for your redesign:
        {strategy}

        Now output ONE new prompt template following the STRATEGY and the hard rules.
        Remember: output ONLY the template text, and it MUST contain {{fulltext}}.
    """).strip()
    messages=[{"role":"system","content":PATCH_SYSTEM},{"role":"user","content":user_msg}]
    text=build_chat_text(gen_tok, messages)
    inputs=gen_tok([text], return_tensors="pt", truncation=True, max_length=4096).to(gen_mdl.device)
    with torch.no_grad():
        out=gen_mdl.generate(**inputs, max_new_tokens=512, temperature=0.9, top_p=0.95,
                             do_sample=True, pad_token_id=gen_tok.pad_token_id, eos_token_id=gen_tok.eos_token_id)
    raw=gen_tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    return _normalize_prompt(raw)

def run_patch(current_prompt, critique, gen_tok, gen_mdl, n_candidates):
    """Generate n_candidates via SEPARATE calls (one per strategy).
    Each is normalised so {fulltext} is guaranteed. Retries once per slot on
    failure; if still bad, falls back to the normalised current prompt so the
    iteration always ends with exactly n_candidates."""
    candidates = []
    for k in range(n_candidates):
        strategy = PATCH_STRATEGIES[k % len(PATCH_STRATEGIES)]
        cand = ""
        for attempt in range(2):
            cand = _generate_one_candidate(current_prompt, critique, strategy, gen_tok, gen_mdl)
            # accept if it has the placeholder and isn't trivially short / identical
            if "{fulltext}" in cand and len(cand) >= 40 and cand.strip() != current_prompt.strip():
                break
        if "{fulltext}" not in cand or len(cand) < 40:
            cand = _normalize_prompt(current_prompt)   # guaranteed-valid fallback
            print(f"    [candidate {k+1}] fell back to current prompt (model output unusable)")
        candidates.append(cand)
    print(f"  Patch generated {len(candidates)} candidate prompts")
    return candidates

In [ ]:
# ── Cell 10 · Phase 1 — APO loop (with prompt grammar) ───────
def make_prompt_record(phase, iteration, candidate, prompt, scores, prompt_grammar, critique=""):
    return {"phase":phase, "iteration":iteration, "candidate":candidate,
            "prompt":prompt, "critique":critique,
            "mean_rouge1":round(scores["mean_rouge1"],4), "mean_rouge2":round(scores["mean_rouge2"],4),
            "mean_rouge_l":round(scores["mean_rouge_l"],4), "mean_bert_f1":round(scores["mean_bert_f1"],4),
            "mean_reward":round(scores["mean_reward"],4),           # = summarisation score
            "prompt_grammar":prompt_grammar}

def run_phase1(initial_prompt, apo_pairs, gen_tok, gen_mdl, cfg):
    prompt_history=[]; best_overall_prompt=initial_prompt; best_overall_reward=-1.0
    best_overall_grammar=None; current_prompt=initial_prompt
    print("\n"+"="*60+"\nPHASE 1 — APO LOOP\n"+"="*60)

    print("\n[Iteration 0] Scoring initial prompt on 150 pairs ...")
    scores = evaluate_prompt(current_prompt, apo_pairs, gen_tok, gen_mdl, cfg)
    pg = get_llm_grammar_score(current_prompt, gen_tok, gen_mdl)
    print(f"  ROUGE-L={scores['mean_rouge_l']:.4f}  BERT-F1={scores['mean_bert_f1']:.4f}  "
          f"reward={scores['mean_reward']:.4f}  prompt_grammar={pg}")
    prompt_history.append(make_prompt_record("phase1",0,0,current_prompt,scores,pg,""))
    if scores["mean_reward"]>best_overall_reward:
        best_overall_reward=scores["mean_reward"]; best_overall_prompt=current_prompt; best_overall_grammar=pg

    for it in range(1, cfg.n_iterations+1):
        print(f"\n[Iteration {it}/{cfg.n_iterations}]")
        print("  Running critique ...")
        critique = run_critique(current_prompt, scores, gen_tok, gen_mdl)
        print(f"  Critique snippet: {critique[:160]} ...")
        print("  Running patch instruction ...")
        candidates = run_patch(current_prompt, critique, gen_tok, gen_mdl, cfg.n_candidates)
        if not candidates:
            print("  No valid candidates; keeping current prompt"); continue
        best_iter_reward=-1.0; best_iter_prompt=current_prompt; best_iter_scores=scores
        for idx,cand in enumerate(candidates):
            print(f"  Scoring candidate {idx+1}/{len(candidates)} ...")
            s = evaluate_prompt(cand, apo_pairs, gen_tok, gen_mdl, cfg)
            cpg = get_llm_grammar_score(cand, gen_tok, gen_mdl)
            print(f"    ROUGE-L={s['mean_rouge_l']:.4f}  BERT-F1={s['mean_bert_f1']:.4f}  "
                  f"reward={s['mean_reward']:.4f}  prompt_grammar={cpg}")
            prompt_history.append(make_prompt_record("phase1",it,idx+1,cand,s,cpg,critique))
            if s["mean_reward"]>best_iter_reward:
                best_iter_reward=s["mean_reward"]; best_iter_prompt=cand; best_iter_scores=s
            if s["mean_reward"]>best_overall_reward:
                best_overall_reward=s["mean_reward"]; best_overall_prompt=cand; best_overall_grammar=cpg
                print(f"    New overall best reward: {best_overall_reward:.4f}")
        current_prompt=best_iter_prompt; scores=best_iter_scores
        print(f"  Best this iteration: reward={best_iter_reward:.4f}")
    print(f"\n=== Phase 1 complete. Overall best reward: {best_overall_reward:.4f} ===")
    return best_overall_prompt, prompt_history, best_overall_reward, best_overall_grammar

In [ ]:
# ── Cell 11 · Phase 2 — Full evaluation (592 pairs) ──────────
def run_phase2(best_prompt, all_pairs, gen_tok, gen_mdl, nli_tok, nli_mdl, nli_device, cfg):
    print("\n"+"="*60+"\nPHASE 2 — FULL EVALUATION\n"+"="*60)
    filenames, fulltexts, references, generated = [],[],[],[]
    gen_complete = []
    for i,pair in enumerate(all_pairs):
        print(f"  Generating {i+1}/{len(all_pairs)} ...", end="\r")
        g, gtc, ic = generate_summary(pair["fulltext"], pair["summary"], best_prompt, gen_tok, gen_mdl, cfg)
        filenames.append(pair["filename"]); fulltexts.append(pair["fulltext"])
        references.append(pair["summary"]); generated.append(g); gen_complete.append(ic)
    print("\n  Generation complete.")

    print("  Computing ROUGE ...")
    rouge_rows=[compute_rouge(g,r) for g,r in zip(generated, references)]
    bert_device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"  Computing BERTScore on {bert_device} ...")
    bs = compute_bertscore_batch(generated, references, cfg, device=bert_device)
    print("  Computing entailment (doc -> reference) ...")
    ent_ref = compute_entailment_batch(fulltexts, references, nli_tok, nli_mdl, nli_device)
    print("  Computing entailment (doc -> generated) ...")
    ent_gen = compute_entailment_batch(fulltexts, generated, nli_tok, nli_mdl, nli_device)
    print("  Computing summary grammar (same model) ...")
    gram = [get_llm_grammar_score(g, gen_tok, gen_mdl) for g in generated]

    rows=[]
    for i in range(len(all_pairs)):
        er, eg = ent_ref[i], ent_gen[i]
        rl = round(rouge_rows[i]["rougeL"],4); bf1 = round(bs["bert_f1"][i],4)
        rows.append({
            "filename": filenames[i],
            "reference_summary": references[i],
            "generated_summary": generated[i],
            "entailment_prob_reference":    round(er["entailed_prob"],4),
            "neutral_prob_reference":       round(er["neutral_prob"],4),
            "contradiction_prob_reference": round(er["contradiction_prob"],4),
            "reference_entailed":           er["label"],
            "entailment_prob_generated":    round(eg["entailed_prob"],4),
            "neutral_prob_generated":       round(eg["neutral_prob"],4),
            "contradiction_prob_generated": round(eg["contradiction_prob"],4),
            "generated_entailed":           eg["label"],
            "both_entailed": (er["label"]=="entailment" and eg["label"]=="entailment"),
            "rouge1": round(rouge_rows[i]["rouge1"],4),
            "rouge2": round(rouge_rows[i]["rouge2"],4),
            "rougeL": rl,
            "bert_precision": round(bs["bert_precision"][i],4),
            "bert_recall":    round(bs["bert_recall"][i],4),
            "bert_f1": bf1,
            "grammar_score": gram[i],
            "summarisation_score": round(W_ROUGEL*rl + W_BERT*bf1, 4),
            "within_budget": True,
            "is_complete_sentence": gen_complete[i],
        })
    return pd.DataFrame(rows)

In [ ]:
# ── Cell 12 · Graphs (top/bottom 10) ─────────────────────────
def plot_top_bottom(df, score_col, color_hi, color_lo, title_hi, title_lo, path_hi, path_lo):
    x = df.copy(); x[score_col]=pd.to_numeric(x[score_col], errors="coerce"); x=x.dropna(subset=[score_col])
    for subset,color,title,path,asc in [
        (x.nlargest(10, score_col), color_hi, title_hi, path_hi, False),
        (x.nsmallest(10, score_col), color_lo, title_lo, path_lo, True)]:
        s=subset.sort_values(score_col, ascending=asc)
        names=[f.replace("multiclinsum_gs_en_","case_") for f in s["filename"]]; vals=s[score_col].tolist()
        fig,ax=plt.subplots(figsize=(13,6))
        bars=ax.bar(range(len(names)), vals, color=color, edgecolor="black", linewidth=0.7, alpha=0.85)
        for b,v in zip(bars,vals): ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.002, f"{v:.4f}", ha="center", va="bottom", fontsize=9, fontweight="bold")
        ax.set_xticks(range(len(names))); ax.set_xticklabels(names, rotation=45, ha="right", fontsize=9)
        lo,hi=min(vals),max(vals); pad=max(0.01,(hi-lo)*0.15)
        ax.set_ylim(max(0,lo-pad), min(1.0,hi+pad)); ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.2f"))
        ax.set_title(f"{title}\n({cfg.model_display} APO)", fontsize=13, fontweight="bold", pad=15)
        ax.grid(axis="y", linestyle="--", alpha=0.5); ax.set_axisbelow(True)
        plt.tight_layout(); plt.savefig(path, dpi=150, bbox_inches="tight"); plt.close(); print("[SAVED]", path)

def generate_all_graphs(df, cfg):
    plot_top_bottom(df, "rougeL", "#2ecc71","#e74c3c",
        "Top 10 — Highest ROUGE-L","Top 10 — Lowest ROUGE-L", OUT("top10_high_rougeL.png"), OUT("top10_low_rougeL.png"))
    plot_top_bottom(df, "bert_f1", "#3498db","#e67e22",
        "Top 10 — Highest BERTScore F1","Top 10 — Lowest BERTScore F1", OUT("top10_high_bert_f1.png"), OUT("top10_low_bert_f1.png"))

In [ ]:
# ── Cell 13 · Save outputs ───────────────────────────────────
def save_outputs(df, prompt_history, best_prompt, best_reward, best_prompt_grammar, cfg):
    results_path = OUT("apo_final_results.csv"); df.to_csv(results_path, index=False)
    print(f"Saved -> {results_path} ({len(df)} rows)")
    ph_path = OUT("apo_prompt_history.json")
    with open(ph_path,"w",encoding="utf-8") as f: json.dump(prompt_history, f, indent=2, ensure_ascii=False)
    print(f"Saved -> {ph_path} ({len(prompt_history)} entries)")

    avg_rougeL  = round(df["rougeL"].mean(),4)
    avg_bert_f1 = round(df["bert_f1"].mean(),4)
    avg_grammar = round(df["grammar_score"].mean(),2)
    summ_score  = round(W_ROUGEL*avg_rougeL + W_BERT*avg_bert_f1, 4)
    metrics = {
        "model": cfg.model_tag, "model_repo": cfg.generator_model, "method": "apo",
        "n_cases": int(len(df)), "fixed_tokens": cfg.fixed_tokens,
        "avg_rougeL": avg_rougeL, "avg_bert_f1": avg_bert_f1,
        "avg_summary_grammar_score": avg_grammar,
        "best_prompt_grammar_score": best_prompt_grammar,
        "summarisation_score": summ_score,
        "best_prompt_reward": round(best_reward,4),
        "best_prompt": best_prompt,
    }
    with open(OUT("apo_metrics.json"),"w") as f: json.dump(metrics, f, indent=2)

    print("\n"+"="*60)
    print(f"  APO FINAL METRICS — {cfg.model_display} ({len(df)} cases)")
    print("="*60)
    print(f"  Average ROUGE-L        : {avg_rougeL}")
    print(f"  Average BERTScore (F1) : {avg_bert_f1}")
    print(f"  Avg summary grammar    : {avg_grammar} / 100")
    print(f"  Best prompt grammar    : {best_prompt_grammar} / 100")
    print(f"  Summarisation score    : {summ_score}  (= {W_ROUGEL}*ROUGE-L + {W_BERT}*BERT-F1)")
    print(f"  Both entailed          : {df['both_entailed'].mean()*100:.1f}%")
    print(f"  Complete sentences     : {df['is_complete_sentence'].mean()*100:.1f}%")
    print("="*60)

In [ ]:
# ── Cell 14 · Initial prompt & run ───────────────────────────
# NOTE: word-count line removed (fixed 150-token regime, matches ISP/zero-shot).
initial_prompt = (
    "Write a concise clinical summary of the following case report in clear, continuous prose.\n"
    "Cover all of these points without blank lines between them:\n"
    "1. Patient demographics (age, sex)\n"
    "2. Presenting complaint and relevant medical history\n"
    "3. Key clinical findings and investigations\n"
    "4. Diagnosis\n"
    "5. Treatment plan\n"
    "6. Clinical outcome\n"
    "Use clear, professional medical language.\n\n"
    "Clinical case:\n{fulltext}\n\nSummary:"
)

all_pairs = load_pairs(cfg)
apo_pairs = select_apo_pairs(all_pairs, cfg.n_apo_pairs, cfg.seed)
gen_tok, gen_mdl = load_generator(cfg)

best_prompt, prompt_history, best_reward, best_prompt_grammar = run_phase1(
    initial_prompt, apo_pairs, gen_tok, gen_mdl, cfg)
print(f"\nBest prompt (reward={best_reward:.4f}, grammar={best_prompt_grammar}):\n{'-'*60}\n{best_prompt}\n{'-'*60}")

nli_tok, nli_mdl, nli_device = load_nli(cfg)
df_results = run_phase2(best_prompt, all_pairs, gen_tok, gen_mdl, nli_tok, nli_mdl, nli_device, cfg)
generate_all_graphs(df_results, cfg)
save_outputs(df_results, prompt_history, best_prompt, best_reward, best_prompt_grammar, cfg)
print("\nDone —", cfg.model_tag)